# [0] Naive Trial with MaxPooling with EXAONE

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import BalancedSWUnivDaconDataset, BalancedDataLoader

from transformers import PreTrainedModel, AutoModelForCausalLM, AutoConfig, AutoTokenizer
from torch.utils.data import DataLoader
from torch import nn, optim
import torch

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import gc
import re

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

In [ ]:
# project name
PROJECT_NAME = "0_naive_maxpool"

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device
device_num = 0

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1, balancing_ratio=1)
valid_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1, balancing_ratio=1)
test_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

#### Paragraph splitting test

In [33]:
def document_to_paragraphs(doc):
    result = doc.split("\n\n")
    if len(result) > 1:
        return [res.strip() for res in result]
    result = doc.split("\n")
    if len(result) > 1:
        return [res.strip() for res in result]
    if len(doc) > 650:
        sentences = re.split(r'(?<=[.?!])\s*', doc)
        sentences = [s.strip() for s in sentences if s]
        return [" ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]  # group by 3 sentences
    else:
        return [doc.strip()]

In [34]:
converted = [(document_to_paragraphs(document[0]), document[1]) for document in train_dataset]
conv_size = []
for document, label in converted:
    conv_size.append(len(document))
    if len(document) == 1:
        print(f"ERROR: Document with single paragraph - {document[0]}", file=sys.stderr, flush=True)
    elif len(document) > 50:
        print(f"WARNING: Document with too many paragraphs - {len(document)} paragraphs", flush=True)
    else:
        pass

ERROR: Document with single paragraph - 1990년 태평양 돌핀스 시즌은 태평양 돌핀스가 KBO 리그에 참가한 3번째 시즌으로, 삼미 슈퍼스타즈, 청보 핀토스 시절까지 합하면 9번째 시즌이다. 김성근 감독이 팀을 이끈 마지막 시즌이며 김성근 감독이 시즌 전 소위 "임호균 각서 파동" 때문에 구단 최고책임자와 감정싸움을 벌인 데다 5할 승률을 목표로 4위에 턱걸이한다는 전략을 세웠으나 1989년 포스트시즌 진출의 여파로 최창호 정명원 김동기 등이 전지훈련에 합류하지 못할 정도로 긴 연봉싸움을 벌인 것 외에도 에이스 박정현마저 허리부상으로 도중하차하는 바람에 전년도의 돌풍을 더 이상 이어가지 못한 데다 해태와의 7월 경기에서 밀린 뒤 0. 496(58승 59패 3무)에 그쳤으며 7팀 중 정규시즌 5위에 머물러 포스트시즌 진출에 실패했고 이로 인해 1988년 9월 10일부터 3년 계약으로 취임했던 김성근 감독이 계약기간을 1년 남겨둔 채 물러났지만 그 해 58승으로 1989년 단일시즌제 도입 후(양대리그 체제인 99~2000년 제외) 2015년 와일드카드 제도 도입 이전까지의 5위팀 최다 승 기록(종전 기록은 89년 OB 54승)을 갱신했으나 1993년 빙그레(61승)(롯데(62승)에 비해 승수에서 뒤졌음에도 패수(빙그레 61 롯데 63)에서 우세) 1998년 해태(61승)(승수는 OB와 같았지만 패수에서 뒤짐)에 의해 갱신되는 듯 했으나 2002년 두산이 66승으로 갱신했지만(LG와 승수가 같았음에도 패수에서 뒤짐) 2013년 롯데(66승)에 의해 타이가 됐다.


ERROR: Document with single paragraph - 주희는 장쑤성 롄윈강 현 하이저우 구 출생으로 고향은 저장성 사오싱이었다. 베이징 대학 철학과를 졸업하고 <신조>에 글을 기고하는 등 활발한 활동을 펼쳤으며, 저장성 사범학교에서 국어 교사로 일하기도 했다. 1923년 발표한 시 <훼멸>은 당시 중국 시단에 큰 영향을 미쳤고, 1925년에는 칭화대학교 중문학과 교수가 되었다. 칭화대 재직 시절 쓴 수필 <아버지의 뒷모습>은 아버지와의 이별을 회상하며 쓴 작품으로, 그의 대표작이자 당시 수필계에 큰 반향을 불러일으켰다. 1927년 발표한 <하당월색>에서는 서정적인 분위기 속에 시대의 고뇌와 자유에 대한 갈망을 드러냈다. 1년간 영국 유학을 다녀온 후에는 <유럽 여행 잡기>와 <런던 잡기>를 출간했다. 1932년 칭화대학교 중문과 주임이 된 그는 1937년 중일 전쟁 이후 쿤밍에서 서남 연합대학교 중문과 교수와 도서관장을 역임했다. 경파 문학의 대표적인 인물이었던 그는 절친이자 동료였던 원이둬가 국민당의 암살로 죽자 큰 충격을 받고 그의 유고를 정리하기도 했다. 베이핑 함락 직전인 1948년 위궤양으로 요절했다. 높은 학력이나 방대한 학술적 업적은 없었지만, 량치차오와 왕궈웨이 같은 거장들이 있던 칭화대에서 배우고 가르치며 쌓은 학문적 기반과 교육 열정을 묵묵히 지켜나간 대표적인 근대 중국 학자였다.


ERROR: Document with single paragraph - 타이베이에서 유학 중이던 루시는 남자친구 리처드의 속임수에 넘어가 조직 두목 미스터 장에게 서류 가방을 전달하게 됩니다. 호텔에서 미스터 장을 찾는 동안 리처드는 죽고 루시는 폭력배들에게 끌려 호텔 방으로 갑니다. 정신을 잃은 사이 루시의 배를 갈라 마약 성분의 CPH4가 담긴 비닐봉투를 넣고 꿰매 버립니다. 미국으로 운반하라는 명령과 함께. 미국행을 기다리던 루시는 감금당하고 감시원에게 저항하다 배를 차여 CPH4가 터져 나옵니다. 그 결과 루시의 뇌 활용도가 높아지며 놀라운 능력이 생겨납니다. 뇌 활용률 24%에선 신체를 완벽하게 제어하고 40%가 되면 모든 상황을 제어할 수 있게 됩니다. 62%에 이르자 타인의 행동까지 조종할 수 있죠. 결국 감금된 곳에서 탈출한 루시는 전사가 되어 미스터 장에게 복수하고 뇌 과학자 노먼 교수에게 도움을 요청합니다. CPH4를 더 투여할수록 뇌 활용률은 높아지지만 인간성은 사라져 갑니다. 결국 루시는 자신도 통제할 수 없는 존재가 되어 '신'과 같은 존재가 되어 자신의 지식을 USB에 담아 전하고 사라집니다. 미스터 장은 경찰 피에르 델 리오에게 죽임을 당하고요. 그리고 사라진 루시는 과거의 자신과 만나 뇌 활용률 100%를 달성합니다.


ERROR: Document with single paragraph - 소송물 논쟁은 소송상의 청구 내용이 되는 권리주장을 개별화하는 기준을 무엇에서 구하느냐에 따른 논쟁이며, 이는 청구를 개별화하는 데 있어 불가피합니다. 지금까지의 이론에서는 권리주장의 내용이 소유권이나 임차권과 같은 실체법상의 개별적인 권리라고 여겨왔습니다. 따라서 법원은 이러한 개별적인 권리의 존부를 판단하면 되고, 판단의 효력도 그 권리의 범위에서 그치는 것으로 해석해 왔습니다. 그러나 최근 대두된 새로운 학설에서는 분쟁해결의 목적에 맞도록 실체법상의 개별적인 권리로부터 벗어나, 분쟁의 근원이 되는 사실관계에 초점을 맞추어 결정해야 한다고 설명하고 있습니다. 전자의 경우 이론적으로는 훌륭하지만, 1회의 재판으로 현실의 분쟁을 해결하는 것이 민사소송의 역할이라는 소박한 의문에 답하지 못하는 약점이 있습니다. 후자는 분쟁사실을 직시하고 소송목적의 차원에서 이론을 도출한다는 점에서 분쟁해결이라는 목적관에 부합하지만, 사실관계를 정확하게 파악한다는 면에서 전자의 설에 일부 양보하는 경우가 많습니다. 그러나 이러한 양보는 다른 이론구성으로 보충할 수 있다는 점을 고려하면, 민사소송의 목적관에 보다 가까운 후자의 학설이 미래를 약속받을 것이라는 데 의문의 여지가 없습니다.


ERROR: Document with single paragraph - 그는 왕족이었고, 특히 그의 친가는 겹사돈으로 유명한 집안이었습니다. 왕족 남자들은 서로 형수, 처제, 삼촌, 이모부, 숙모, 이모 등의 복잡한 친인척 관계를 맺고 있었죠. 그는 619년, 7살 연상의 부인과 결혼해 8남매를 두었습니다. 하지만 634년에 첫째 딸과 두 아들을 콜레라로 잃는 큰 슬픔을 겪기도 했습니다. 640년 4월 12일에는 사촌 형인 다토파 티사 1세가 왕이 되는 것을 보았고, 646년 그에게 후계자로 책립되었습니다. 650년 4월 12일, 마침내 왕위를 물려받아 42세의 나이로 왕이 되었죠. 이듬해에는 외가 친척인 다풀라 공을 후계자로 삼았습니다. 그는 인자하고 온화한 정치를 펼쳤지만, 658년부터 장결핵을 앓게 되어 다풀라 공에게 국정을 맡겼습니다. 결국 병을 이기지 못하고 659년 9월 15일, 51세의 나이로 세상을 떠났습니다. 그의 죽음 후, 다풀라 공은 2년간 대리 통치를 하다가 661년 9월 15일에 왕이 되었습니다.


ERROR: Document with single paragraph - 리본은 히트맨이자 주인공인 츠나의 가정교사다. 그는 검은색 정장의 마피아 복장을 즐겨 입고, 모자에는 파트너 레온(카멜레온)을 데리고 다닌다. 일반인들에게는 상냥한 모습을 보이지만, 패밀리 관계자나 츠나에게는 장난스러우면서도 엄격한 태도를 보인다. 그의 애인은 최소 4명이라고 하는데, 그중 비앙키가 4번째라고 한다. 전투 시에는 주로 츠나에게 필살탄이나 잔소리탄을 쏘아서 싸우게 한다. 그의 실체는 아르꼬발레노로, '보린'이라는 가명으로 천재 수학자로 활약하기도 했다. 그의 전투 능력은 최상위 수준이지만, 가정교사라는 입장 때문에 직접 싸우지는 않는다. 히바리 쿄야의 일격도 가볍게 받아내는 등 강력한 실력을 보유하고 있다. 츠나에게 있어 리본은 정신적으로 큰 스승이며, 서로의 신뢰관계가 대단하다.


ERROR: Document with single paragraph - 리한(LEEHAN, 理韓)은 1979년 박인철 현 회장이 설립한 자동차부품 전문기업이다. 서진산업에서 분사하여 설립된 대기산업을 모태로 하고 있으며, 박인철 회장의 장남인 박지훈 사장이 대표이사를 맡고 있다. 지주회사인 리한을 중심으로 강소리한, 북경리한, 리한아메리카 등 중국과 미국에 해외생산법인이 있으며, 합작사인 리한브로제가 계열사로 편제되어 있다. 초기에는 자동차 래치(Latch)류가 주요 생산품목이었으며, 기아자동차 '푸조'와 트럭 'T-2000', 'T-600'에 들어가는 도어래치, 잭크 등을 공급하였다. 이후 1982년 에어클리너(Air Cleaner)를 추가로 생산하기 시작했고, 1987년 반월공장을 완공하였다. 이어 1989년 기술연구소 설립, 1992년 시화공장 완공과 더불어 국내를 기반으로 사업영역을 확장했다. 그러나 1997년 7월 기아그룹 부도의 여파로 경영난에 처하게 되자, 계열회사 구조조정을 진행하며 경영정상화에 힘썼다. 이후 중국과 미국에 공장을 연이어 완공하고, 해외 기업들과 전략적 기술제휴를 체결하며 글로벌 기업으로 성장해 나갔다. 2011년 9월에는 대기산업에서 리한으로 사명을 변경했다.


## Define Model

In [ ]:
base_model_id = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

In [ ]:
class ExaoneForNaiveTextDetection(PreTrainedModel):
    def __init__(
        self
    ):
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto"
        )
        super().__init__(base.config)

        # 0. Register base model
        self.base = base.transformer
        for param in self.base.parameters():
            param.requires_grad = False  # Freeze the base model parameters

        # 1. Add MaxPooling layer
        self.pool = nn.AdaptiveAvgPool1d(1)

        # 2. Final classification layer
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.config.hidden_size, 1),
        )

        # 3. Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids: list[torch.LongTensor],  # paragraph of sentences
        attention_mask: list[torch.Tensor]  # attention mask for each paragraph
    ) -> torch.Tensor:
        hidden_states = []
        with torch.no_grad():
            for inputs, masks in zip(input_ids, attention_mask):
                processed = self.base(input_ids=inputs, attention_mask=masks).last_hidden_state
                hidden_states.append(processed)

        hidden_states = torch.cat(hidden_states, dim=1)
        if self.training:
            pooled = self.pool(hidden_states)
            return self.classifier(pooled)
        else:
            return self.classifier(hidden_states)

In [ ]:
model = ExaoneForNaiveTextDetection()
try:
    from safetensors.torch import load_file
    state_dict = load_file(f"./models/{PROJECT_NAME}_last/model.safetensors")
    model.load_state_dict(state_dict)
except Exception:
    pass
model.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(device)

In [ ]:
tokenize("Hello, my dog is cute")

## Train and Evaluate

### Utils

In [35]:
def to_label(scores, threshold=0.5):
    return [1 if score > threshold else 0 for score in scores]

def calc_score(lb, pd):
    pd_discrete = to_label(pd)
    acc = accuracy_score(lb, pd_discrete)
    f1 = f1_score(lb, pd_discrete)
    rocauc = roc_auc_score(lb, pd)
    return f"ACC: {acc:.6f}, F1: {f1:.6f}, ROCAUC: {rocauc:.6f}"

In [ ]:
BATCH_SIZE = 1, 1, 1
GRADIENT_ACCUMULATION_STEPS = 4  # real batch

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=lambda x: x)

In [ ]:
EPOCHS = 10
LEARNING_RATE = 2e-5, 1e-6

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE[0])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1, min_lr=LEARNING_RATE[1])

### Training Loop

In [ ]:
with (
    tqdm(range(EPOCHS), desc="[Running Epochs]") as epochs,
    tqdm(range(len(train_dataset)//BATCH_SIZE[0]), desc="[Training]") as train_progress,
    tqdm(range(len(valid_dataset)//BATCH_SIZE[1]), desc="[Validating]") as valid_progress
):
    for epoch in epochs:
        train_progress.reset()
        train_loss, train_preds, train_labels = [], [], []

        # Train
        model.train()
        for step, (texts, labels) in enumerate(train_loader):
            texts, labels = texts[0], labels[0]  # Unpack single batch
            try:
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in document_to_paragraphs(texts)]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                loss = criterion(logits, labels.float().to(device)) #* (1.4 if labels[0].item() == 1 else 1)
                train_preds.append(torch.sigmoid(logits)[0][0].item())
                train_labels.append(labels.item())
                train_loss.append(loss.item())
                loss.backward()

                if (step+1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad()

                train_progress.update(1)
                train_progress.set_description(f"[Training] Step: {step+1}, Loss: {sum(train_loss)/len(train_loss):.6f}," + calc_score(train_labels, train_preds))
            except Exception as e:
                print(e, file=sys.stderr)

        # Validate
        model.eval()
        valid_loss, valid_preds, valid_labels = [], [], []
        torch.cuda.empty_cache(); gc.collect(); valid_progress.reset()
        for texts, labels in valid_loader:
            texts, labels = texts[0], labels[0]  # Unpack single batch
            try:
                with torch.no_grad():
                    input_ids, attention_masks = [], []
                    for tokenized in [tokenize(t) for t in document_to_paragraphs(texts)]:
                        input_ids.append(tokenized['input_ids'].to(device))
                        attention_masks.append(tokenized['attention_mask'].to(device))

                    logits = model(input_ids=input_ids, attention_mask=attention_masks)
                    loss = criterion(logits, labels.float().to(device)) #* (1.4 if labels[0].item() == 1 else 1)
                    scores = torch.sigmoid(logits)[0]
                    preds = to_label(scores)

                    valid_preds.append(scores[0].item())
                    valid_labels.append(labels.item())
                    valid_loss.append(loss)
            except Exception as e:
                print(e, file=sys.stderr)

            valid_progress.update(1)
            valid_progress.set_description(f"[Validating] Loss: {torch.mean(torch.stack(valid_loss)):.6f}, " + calc_score(valid_labels, valid_preds))

        model.save_pretrained(f"./models/{PROJECT_NAME}_{epoch}")
        model.save_pretrained(f"./models/{PROJECT_NAME}_last")
        scheduler.step(torch.mean(torch.stack(valid_loss)))

In [ ]:
# Model Saving
model.save_pretrained(f"./models/{PROJECT_NAME}_last")

### Final Output

In [ ]:
per_titles = {}
for i, row in test_dataset.raw.iterrows():
    if row['title'] not in per_titles:
        per_titles[row['title']] = [row['paragraph_text']]
    else:
        per_titles[row['title']].append(row['paragraph_text'])
test_dataset_bundled = list(per_titles.values())
test_dataset_bundled

In [ ]:
results = []
with tqdm(test_dataset_bundled, desc="[Testing]") as progress:
    humans, ais = 0, 0
    model.eval()
    for texts in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in texts]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                scores = torch.sigmoid(logits)
                results.extend(scores.tolist())
                for preds in to_label(scores):
                    if preds == 0:
                        humans += 1
                    else:
                        ais += 1
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Testing] Human: {humans/len(test_dataset):.2%}, Ai: {ais/len(test_dataset):.2%}")

len(results) == len(test_dataset)

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x="generated", kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv(f"./data/submission_{PROJECT_NAME[2:]}.csv", index=False, encoding='utf-8-sig')